# 工具调用
## 定义
模型可以请求调用工具来执行诸如从数据库获取数据、搜索网络或运行代码等任务。
工具是以下各项的组合：
* 模式，包括工具名称、描述和/或参数定义（通常是 JSON 模式）
* 函数或协程执行。

## 使用
要使你定义的工具可供模型使用，你必须使用绑定方法将它们绑定起来 bind_tools。
在后续调用中，模型可以根据需要选择调用任何已绑定的工具。

## 工具执行循环
绑定用户自定义工具时，模型的响应包含执行该工具的请求，如下执行所示：

In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,
)

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Tool: get_weather
Args: {'location': 'Boston'}


如果模型 model 与代理 agent 分开使用，则需要自行执行请求的工具并将结果返回给模型以供后续推理使用.
如果使用代理 agent ，代理循环将自动处理工具执行循环。

In [2]:
from langchain.messages import HumanMessage
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [HumanMessage(content="What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


## 累计工具调用

In [ ]:
gathered = None
for chunk in model_with_tools.stream("What's the weather in Boston?"):
    gathered = chunk if gathered is None else gathered + chunk
    print(gathered.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '2bfaf45f-a835-4710-b7c5-56d0d580ae17', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '2bfaf45f-a835-4710-b7c5-56d0d580ae17', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '2bfaf45f-a835-4710-b7c5-56d0d580ae17', 'type': 'tool_call'}]
